## ↓のデータフレーム変換関数を定義したセルを実行済みの状態で進めてください

In [ ]:
import requests
import time
import pandas as pd

def gen_boj_dataframe(
    db:str,
    codes:list[str],
    startdate:str,
    enddate:str) -> pd.DataFrame:
    url = "https://www.stat-search.boj.or.jp/api/v1/getDataCode"
    params = {
        "DB": db,
        "CODE": ",".join(codes),
        "FORMAT": "JSON",
        "LANG":"JP",
        "STARTDATE": startdate,
        "ENDDATE": enddate
        }

    # ページネーションで最後のページを取得するまで、繰り返し処理を実行
    # Web APIからの取得データを格納
    all_results = []

    while True:
        # Web APIへのリクエストと例外処理。エラー時は空のデータフレームを返して終了
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            response_data = response.json()
        except requests.exceptions.Timeout:
            print("Web APIから30秒以内に応答がありませんでした。")
            return pd.DataFrame()
        except requests.exceptions.ConnectionError:
            print("Web APIに接続できませんでした。")
            return pd.DataFrame()
        except requests.exceptions.HTTPError as error:
            print(f"HTTPエラーが発生しました: {error}")

            content_type = response.headers.get("Content-Type", "")
            if "application/json" in content_type:
                error_data = response.json()
                print(error_data["MESSAGE"])
            return pd.DataFrame()
        except requests.exceptions.JSONDecodeError:
            print("レスポンスをJSONとして読み込めませんでした。")
            return pd.DataFrame()
        else:
            if response_data["MESSAGEID"] == "M181030I":
                print(response_data["MESSAGE"])
                return pd.DataFrame()

            all_results.extend(response_data["RESULTSET"])
            next_position = response_data["NEXTPOSITION"]
            if next_position is None:
                break

            params["STARTPOSITION"] = next_position
            time.sleep(1)

    # 取得結果をデータフレームに変換
    df = pd.DataFrame(all_results)
    result_df = pd.DataFrame()

    for i in range(len(df)):
        values_df = pd.DataFrame(df.loc[i,"VALUES"])
        values_df = values_df.assign(
            SERIES_CODE=df.loc[i, "SERIES_CODE"],
            NAME_OF_TIME_SERIES_J=df.loc[i, "NAME_OF_TIME_SERIES_J"],
            CATEGORY_J=df.loc[i, "CATEGORY_J"]
        )
        result_df = pd.concat([result_df, values_df])

    result_df = result_df[["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J","SURVEY_DATES","VALUES"]]

    result_df = result_df.reset_index(drop=True)

    result_df = result_df.astype({'SURVEY_DATES': str})

    return result_df

## 企業物価指数・企業向けサービス価格指数の推移を折れ線グラフにしよう

In [ ]:
import plotly.express as px

# 企業物価指数の系列コードの取得
url = "https://www.stat-search.boj.or.jp/api/v1/getMetadata"

response_pr01 = requests.get(url, params={"DB":"PR01"})

data_pr01 = response_pr01.json()

pr01_series_data = [
    {
        "SERIES_CODE": item["SERIES_CODE"],
        "NAME_OF_TIME_SERIES_J": item["NAME_OF_TIME_SERIES_J"],
        "CATEGORY_J": item["CATEGORY_J"],
    }
    for item in data_pr01.get("RESULTSET", [])
    if item.get("SERIES_CODE")
]

pr01_series_codes_df = pd.DataFrame(pr01_series_data)

# 企業向けサービス価格指数の系列コードの取得
response_pr02 = requests.get(url, params={"DB":"PR02"})

data_pr02 = response_pr02.json()

pr02_series_data = [
    {
        "SERIES_CODE": item["SERIES_CODE"],
        "NAME_OF_TIME_SERIES_J": item["NAME_OF_TIME_SERIES_J"],
        "CATEGORY_J": item["CATEGORY_J"],
    }
    for item in data_pr02.get("RESULTSET", [])
    if item.get("SERIES_CODE")
]

pr02_series_codes_df = pd.DataFrame(pr02_series_data)

# 条件に合う系列コードの抽出
pr01_series_codes_df = pr01_series_codes_df.query('(CATEGORY_J == "企業物価指数 2020年基準/国内企業物価指数") & (NAME_OF_TIME_SERIES_J.str.startswith("類別/"))')
pr02_series_codes_df = pr02_series_codes_df.query('(CATEGORY_J == "企業向けサービス価格指数 2020年基準") & (NAME_OF_TIME_SERIES_J.str.startswith("類別/"))')

# 重複する系列コードの除去
pr01_series_codes = pr01_series_codes_df["SERIES_CODE"].drop_duplicates().tolist()
pr02_series_codes = pr02_series_codes_df["SERIES_CODE"].drop_duplicates().tolist()

# 条件に合う系列コードのデータを取得
pr01_df = gen_boj_dataframe("PR01", pr01_series_codes, "202504", "202607")
pr02_df = gen_boj_dataframe("PR02", pr02_series_codes, "202504", "202607")

# データの結合
price_index_df = pd.concat([pr01_df, pr02_df])
price_index_df = price_index_df.reset_index(drop=True)

# データを横長形式に変換
wide_price_df = price_index_df.pivot(
    index=[
        "SERIES_CODE",
        "NAME_OF_TIME_SERIES_J",
        "CATEGORY_J",
    ],
    columns="SURVEY_DATES",
    values="VALUES"
).reset_index()

# 統計値の上位・下位5位を抽出して結合
top_5_df = wide_price_df.sort_values("202607", ascending=False).head(5)
bottom_5_df = wide_price_df.sort_values("202607", ascending=True).head(5)
top_bottom_wide_df = pd.concat([top_5_df, bottom_5_df]).reset_index(drop=True)

# Plotlyで年月を横軸、指数値を縦軸に指定できるように変換
top_bottom_df = top_bottom_wide_df.melt(id_vars=["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J"], var_name="SURVEY_DATES", value_name="VALUES")

# グラフの表示
fig = px.line(
    top_bottom_df,
    x="SURVEY_DATES",
    y="VALUES",
    color="NAME_OF_TIME_SERIES_J",
    markers=True,
    title="2026年7月時点 国内企業物価指数・企業向けサービス価格指数上位5位・下位5位（2020年基準）",
    labels={
        "SURVEY_DATES": "時期",
        "VALUES": "物価・価格指数",
        "NAME_OF_TIME_SERIES_J": "系列名",
    },
)
fig.show()

## グラフにする類別の基準を変えてみよう

In [ ]:
# 2025年7月から2026年7月までの前年増減率を基準に、上位5系列と下位5系列を選んでグラフ化
import plotly.express as px

# 企業物価指数の系列コードの取得
url = "https://www.stat-search.boj.or.jp/api/v1/getMetadata"

response_pr01 = requests.get(url, params={"DB":"PR01"})

data_pr01 = response_pr01.json()

pr01_series_data = [
    {
        "SERIES_CODE": item["SERIES_CODE"],
        "NAME_OF_TIME_SERIES_J": item["NAME_OF_TIME_SERIES_J"],
        "CATEGORY_J": item["CATEGORY_J"],
    }
    for item in data_pr01.get("RESULTSET", [])
    if item.get("SERIES_CODE")
]

pr01_series_codes_df = pd.DataFrame(pr01_series_data)

# 企業向けサービス価格指数の系列コードの取得
response_pr02 = requests.get(url, params={"DB":"PR02"})

data_pr02 = response_pr02.json()

pr02_series_data = [
    {
        "SERIES_CODE": item["SERIES_CODE"],
        "NAME_OF_TIME_SERIES_J": item["NAME_OF_TIME_SERIES_J"],
        "CATEGORY_J": item["CATEGORY_J"],
    }
    for item in data_pr02.get("RESULTSET", [])
    if item.get("SERIES_CODE")
]

pr02_series_codes_df = pd.DataFrame(pr02_series_data)

# 条件に合う系列コードの抽出
pr01_series_codes_df = pr01_series_codes_df.query('(CATEGORY_J == "企業物価指数 2020年基準/国内企業物価指数") & (NAME_OF_TIME_SERIES_J.str.startswith("類別/"))')
pr02_series_codes_df = pr02_series_codes_df.query('(CATEGORY_J == "企業向けサービス価格指数 2020年基準") & (NAME_OF_TIME_SERIES_J.str.startswith("類別/"))')

# 重複する系列コードの除去
pr01_series_codes = pr01_series_codes_df["SERIES_CODE"].drop_duplicates().tolist()
pr02_series_codes = pr02_series_codes_df["SERIES_CODE"].drop_duplicates().tolist()

# 条件に合う系列コードのデータを取得
pr01_df = gen_boj_dataframe("PR01", pr01_series_codes, "202504", "202607")
pr02_df = gen_boj_dataframe("PR02", pr02_series_codes, "202504", "202607")

# データの結合
price_index_df = pd.concat([pr01_df, pr02_df])
price_index_df = price_index_df.reset_index(drop=True)

# データを横長形式に変換
wide_price_df = price_index_df.pivot(
    index=[
        "SERIES_CODE",
        "NAME_OF_TIME_SERIES_J",
        "CATEGORY_J",
    ],
    columns="SURVEY_DATES",
    values="VALUES"
).reset_index()

# 前年増減率の列を追加
wide_price_df["202607前年増減率"] = (wide_price_df["202607"] - wide_price_df["202507"]) / wide_price_df["202507"]

# 上位・下位の類別5つを抽出して結合
top_5_yoy_df = wide_price_df.sort_values("202607前年増減率", ascending=False).head(5)
bottom_5_yoy_df = wide_price_df.sort_values("202607前年増減率", ascending=True).head(5)
top_bottom_wide_yoy_df = pd.concat([top_5_yoy_df, bottom_5_yoy_df]).reset_index(drop=True)

# .melt()を使い、月ごとの列を`SURVEY_DATES`列と`VALUES`列にまとめ直し
top_bottom_yoy_df = top_bottom_wide_yoy_df.melt(id_vars=["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J"], var_name="SURVEY_DATES", value_name="VALUES")

# 追加した前年増減率の列を除去
top_bottom_yoy_df = top_bottom_yoy_df.query('SURVEY_DATES != "202607前年増減率"')

# グラフ化
fig = px.line(
    top_bottom_yoy_df,
    x="SURVEY_DATES",
    y="VALUES",
    color="NAME_OF_TIME_SERIES_J",
    markers=True,
    title="2026年7月対前年増減率 国内企業物価指数・企業向けサービス価格指数上位5位・下位5位（2020年基準）",
    labels={
        "SURVEY_DATES": "時期",
        "VALUES": "物価・価格指数",
        "NAME_OF_TIME_SERIES_J": "系列名",
    },
)
fig.show()